# 01 - Pemeriksaan data (EDA)

Tujuan: memeriksa dataset PhiUSIIL sebelum dipakai: bentuk data, kelas target, nilai hilang, duplikat, kolom Domain, dan fitur yang berpotensi bocor.
Notebook ini hanya membaca data dan tidak mengubah berkas apa pun.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent          # folder repo (notebook ada di folder notebooks/)
DATA_PATH = ROOT / "data" / "PhiUSIIL_Phishing_URL_Dataset.csv"
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)

df = pd.read_csv(DATA_PATH)
print("Bentuk:", df.shape)
df.head()

Bentuk: (235795, 56)


,FILENAME,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS,LineOfCode,LargestLineLength,HasTitle,Title,DomainTitleMatchScore,URLTitleMatchScore,HasFavicon,Robots,IsResponsive,NoOfURLRedirect,NoOfSelfRedirect,HasDescription,NoOfPopup,NoOfiFrame,HasExternalFormSubmit,HasSocialNet,HasSubmitButton,HasHiddenFields,HasPasswordField,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,521848.txt,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,0.061933,3,1,0,0,0.0,18,0.581,0,0.0,0,0,0,1,0.032,1,558,9381,1,à¸‚à¹ˆà¸²à¸§à¸ªà¸” à¸‚à¹ˆà¸²à¸§à¸§à¸±à¸™à¸™à¸µ...,0.000000,0.000000,0,1,1,0,0,0,0,1,0,0,1,1,0,1,0,0,1,34,20,28,119,0,124,1
1,31372.txt,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,0.050207,2,1,0,0,0.0,9,0.391,0,0.0,0,0,0,2,0.087,1,618,9381,1,johannes gutenberg-universitÃ¤t mainz,55.555556,55.555556,1,1,0,0,0,0,0,0,0,1,1,0,0,0,0,0,1,50,9,8,39,0,217,1
2,597387.txt,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,0.064129,2,2,0,0,0.0,15,0.517,0,0.0,0,0,0,2,0.069,1,467,682,1,voice fm southampton,46.666667,46.666667,0,1,1,0,0,1,0,0,0,0,1,1,0,0,0,0,1,10,2,7,42,2,5,1
3,554095.txt,https://www.sfnmjournal.com,26,www.sfnmjournal.com,19,0,com,100.0,1.000000,0.522907,0.057606,3,1,0,0,0.0,13,0.500,0,0.0,0,0,0,1,0.038,1,6356,26824,1,home page: seminars in fetal and neonatal medi...,0.000000,0.000000,0,1,1,0,0,0,1,12,0,1,1,1,0,0,1,1,1,3,27,15,22,1,31,1
4,151578.txt,https://www.rewildingargentina.org,33,www.rewildingargentina.org,26,0,org,100.0,1.000000,0.079963,0.059441,3,1,0,0,0.0,20,0.606,0,0.0,0,0,0,1,0.030,1,6089,28404,1,fundaciÃ³n rewilding argentina,100.000000,100.000000,0,1,1,1,1,1,0,2,0,1,1,1,0,1,1,0,1,244,15,34,72,1,85,1


## 1. Kolom dan tipe data

In [2]:
print(df.dtypes.value_counts())
teks = df.select_dtypes(exclude="number").columns.tolist()
print("Kolom teks/identitas:", teks)

int64      41
float64    10
str         5
Name: count, dtype: int64
Kolom teks/identitas: ['FILENAME', 'URL', 'Domain', 'TLD', 'Title']


## 2. Kelas target
Kolom `label`: 1 = legitimate, 0 = phishing (sesuai deskripsi dataset di Kaggle: 134.850 legitimate dan 100.945 phishing).

In [3]:
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True).round(4))

label
1    134850
0    100945
Name: count, dtype: int64
label
1    0.5719
0    0.4281
Name: proportion, dtype: float64


## 3. Nilai hilang dan duplikat

In [4]:
print("Total nilai hilang:", int(df.isna().sum().sum()))
print("Baris duplikat penuh:", int(df.duplicated().sum()))
print("URL duplikat:", int(df["URL"].duplicated().sum()))

dup = df[df["URL"].duplicated(keep=False)]
label_per_url = dup.groupby("URL")["label"].nunique()
print("Baris yang URL-nya muncul lebih dari sekali:", len(dup))
print("URL duplikat dengan label berbeda:", int((label_per_url > 1).sum()))
print("Kelas pada baris duplikat:")
print(df[df["URL"].duplicated()]["label"].value_counts())

Total nilai hilang: 0
Baris duplikat penuh: 0
URL duplikat: 425
Baris yang URL-nya muncul lebih dari sekali: 850
URL duplikat dengan label berbeda: 0
Kelas pada baris duplikat:
label
0    425
Name: count, dtype: int64


## 4. Kolom Domain
Pembagian per domain (Skenario C) bergantung pada kolom ini. Awalan `www.` dibuang agar `www.contoh.com` dan `contoh.com` masuk satu kelompok.

In [5]:
print(df["Domain"].head(10).tolist())
grup = df["Domain"].str.lower().str.replace(r"^www\.", "", regex=True)
print("Domain berawalan www.:", int(df["Domain"].str.lower().str.startswith("www.").sum()))
print("Domain unik (asli):", df["Domain"].nunique())
print("Domain unik (tanpa www.):", grup.nunique())

ukuran = grup.value_counts()
print("Domain yang hanya punya 1 URL:", int((ukuran == 1).sum()))
print("Baris yang berbagi domain dengan baris lain:", int(ukuran[ukuran > 1].sum()))
print("Domain terbesar:")
print(ukuran.head(5))

label_per_domain = df.assign(grup=grup).groupby("grup")["label"].nunique()
print("Domain dengan label campur:", int((label_per_domain > 1).sum()))

['www.southbankmosaics.com', 'www.uni-mainz.de', 'www.voicefmradio.co.uk', 'www.sfnmjournal.com', 'www.rewildingargentina.org', 'www.globalreporting.org', 'www.saffronart.com', 'www.nerdscandy.com', 'www.hyderabadonline.in', 'www.aap.org']
Domain berawalan www.: 176596
Domain unik (asli): 220086
Domain unik (tanpa www.): 219103
Domain yang hanya punya 1 URL: 213232
Baris yang berbagi domain dengan baris lain: 22563
Domain terbesar:
Domain
ipfs.io                         1197
docs.google.com                  526
mail.deliverylifesupport.com     370
cloudflare-ipfs.com              360
storageapi.fleek.co              318
Name: count, dtype: int64
Domain dengan label campur: 58


## 5. Fitur yang berpotensi bocor
`URLSimilarityIndex` bernilai 100 untuk semua baris legitimate, sehingga dapat membocorkan label.

In [6]:
print(df.groupby("label")["URLSimilarityIndex"].describe().round(2))

          count    mean    std     min     25%     50%     75%    max
label                                                                
0      100945.0   49.62  22.57    0.16   32.12   51.42   67.25  100.0
1      134850.0  100.00   0.00  100.00  100.00  100.00  100.00  100.0


## 6. Kelompok fitur
- **Fitur leksikal URL** (dihitung dari string URL): dipakai di Skenario B dan C.
- **Fitur turunan statistik** (`URLSimilarityIndex`, `CharContinuationRate`, `TLDLegitimateProb`, `URLCharProb`): bukan murni dari string URL.
- **Fitur isi halaman (HTML)**: misalnya `LineOfCode`, `NoOfImage`, `NoOfJS`, `HasSocialNet`.
Daftar lengkap ada di `src/preprocessing.py`.

In [7]:
import sys
sys.path.insert(0, str(ROOT))
from src.preprocessing import URL_ONLY, DERIVED_COLS, TEXT_ID_COLS

html = [c for c in df.select_dtypes("number").columns
        if c not in URL_ONLY + DERIVED_COLS + ["label"]]
print("Fitur URL-only:", len(URL_ONLY))
print("Fitur turunan statistik:", len(DERIVED_COLS))
print("Fitur isi halaman (HTML):", len(html))
print(html)

Fitur URL-only: 18
Fitur turunan statistik: 4
Fitur isi halaman (HTML): 28
['LineOfCode', 'LargestLineLength', 'HasTitle', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef']


## Catatan untuk dijelaskan saat verifikasi
1. Apa arti label 1 dan 0, dan dari mana kita tahu?
2. Mengapa URL duplikat dibuang sebelum membagi data?
3. Mengapa `www.` dibuang dari kolom Domain?
4. Apa arti temuan bahwa hampir semua domain hanya punya satu URL untuk Skenario C?
5. Mengapa `URLSimilarityIndex` dicurigai bocor?